In [1]:
from pathlib import Path

import numpy as np
import scipy.optimize as spo

from cardiac_electrophysiology import posterior_builder as builder
from cardiac_electrophysiology.utils import analysis, visualization

In [2]:
settings = builder.PosteriorBuilderSettings(
    paths=builder.Paths(
        mesh_path=Path("../data/mesh.vtu"),
        basis_vecs_path=Path("../data/basis_vecs.npy"),
        prior_mean_path=Path("../data/prior_mean_from_sde.npy"),
        ground_truth_path=Path("../data/ground_truth_from_sde.npy"),
        log_file_path=Path("../results/lsbip_logfile.log"),
    ),
    prior_parameters=builder.PriorParameters(
        kappa=0.005,
        tau=100,
        seed=0,
    ),
    eikonal_parameters=builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=3,
        transversal_velocity=1,
    ),
    observation_parameters=builder.ObservationParameters(
        num_observations=1000,
        noise_variance=1e-4,
        seed=0,
    ),
    logger_settings=builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)

In [3]:
posterior_builder = builder.PosteriorBuilder(settings)
posterior, additional_output = posterior_builder.build(return_additional_data=True)
visualization.visualize_data_points(
    mesh=additional_output.pv_mesh,
    observation_inds=additional_output.observation_inds,
)

Widget(value='<iframe src="http://localhost:45961/index.html?ui=P_0x7f7137a7f230_0&reconnect=auto" class="pyvi…

In [8]:
initial_guess = np.load("../results/map_estimate.npy")
optimizer_options = {
    "disp": True,
    "maxiter": 2000,
    "ftol": 1e-6,
    "gtol": 1e-6,
    "maxls": 100,
}
map_estimate = spo.minimize(
    fun=posterior.evaluate_cost,
    jac=posterior.evaluate_gradient,
    x0=initial_guess,
    method="L-BFGS-B",
    options=optimizer_options,
)
print(map_estimate)
np.save("../results/map_estimate.npy", map_estimate.x)

  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 14875.538771221527
        x: [ 3.378e-03  3.433e-02 ... -8.074e-01 -2.113e-01]
      nit: 1326
      jac: [ 1.030e+01  2.567e+01 ... -3.592e+01 -5.016e+00]
     nfev: 1435
     njev: 1435
 hess_inv: <15820x15820 LbfgsInvHessProduct with dtype=float64>


In [9]:
map_parameter = np.load("../results/map_estimate.npy")
analysis_data = analysis.compute_map_result_analysis(
    map_parameter=map_parameter,
    posterior=posterior,
    additional_output=additional_output,
)

Prior mean angle L2-error: 47.61438383389681
Prior mean angle max-error: 1.0181236206278492
MAP angle L2-error: 8.944274357708403
MAP angle max-error: 0.31780701808566475
Prior mean predictive L2-error: 270.6304931640625
Prior mean predictive max-error: 8.104975700378418
MAP predictive L2-error: 6.671483039855957
MAP predictive max-error: 0.5103912353515625


In [10]:
for data in (
    analysis_data.prior_mean_parameter,
    analysis_data.ground_truth_parameter,
    analysis_data.map_parameter,
    analysis_data.diff_lat_truth_prior,
    analysis_data.diff_lat_truth_map,
):
    visualization.visualize_scalar_field(
        mesh=additional_output.pv_mesh,
        scalar_field=data,
        circular=False,
    )

Widget(value='<iframe src="http://localhost:45961/index.html?ui=P_0x7f7008130910_6&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:45961/index.html?ui=P_0x7f7137c09310_7&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:45961/index.html?ui=P_0x7f70081316d0_8&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:45961/index.html?ui=P_0x7f7008131f90_9&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:45961/index.html?ui=P_0x7f7008132850_10&reconnect=auto" class="pyv…